In [1]:
import fine as fn
import pyomo.environ as pyomo
import pandas as pd 

# Step 1: Define the Energy System Model
esM = fn.EnergySystemModel(
    locations={"A", "B"},
    onlycommodities={"electricity", "hydrogen"},
    onlycommodityUnitsDict={"electricity": "GW", "hydrogen": "kg"},
    onlymaterials={"steel", "copper"},
    onlymaterialUnitsDict={"steel": "tons", "copper": "kg"}
)

In [2]:
# energyCommoditySet = {'electricty'}
# materialCommoditySet = {'steel'}
# commodityUnitDict = {'electricty': 'kWh', 'steel': 't'}
# print(processedMaterialIntensity[('location1', 2020, 'material1')]) 

# print(processedMaterialIntensity["location1"]["2020"]["material1"]) 

In [3]:
# Check if commodity declarations work
print("only Materials Units Dict:", esM.onlymaterialUnitsDict)
print("only Commodity Units Dict:", esM.onlycommodityUnitsDict)
print("Commodities:", esM.commodities)
print("Commodity Units Dict:", esM.commodityUnitsDict)

only Materials Units Dict: {'steel': 'tons', 'copper': 'kg'}
only Commodity Units Dict: {'electricity': 'GW', 'hydrogen': 'kg'}
Commodities: ['electricity', 'hydrogen', 'copper', 'steel']
Commodity Units Dict: {'electricity': 'GW', 'hydrogen': 'kg', 'copper': 'kg', 'steel': 'tons'}


In [4]:
# Step 2: Add a Energy Source Component that Requires Materials                             
esM.add(
    fn.Source(
        esM=esM, 
        name="Wind Turbines",
        commodity="electricity",
        hasCapacityVariable=True,
        materialIntensity = {
            'A': {
                'steel': pd.Series({0: 3.1,  1: 3.2}, dtype='float64'),
                'copper': pd.Series({0: 5.3, 1: 3.2}, dtype='float64')  
            },
            'B': {
                'steel': pd.Series({0: 2.9, 1: 3.0}, dtype='float64'),
                'copper': pd.Series({0: 4.8, 1: 4.9}, dtype='float64')
                }
            },
        materialRecovery= {
            'A': {
                'steel': pd.Series({0: 3.1,  1: 3.2}, dtype='float64'),
                'copper': pd.Series({0: 5.3, 1: 3.2}, dtype='float64')  
            },
            'B': {
                'steel': pd.Series({0: 2.9, 1: 3.0}, dtype='float64'),
                'copper': pd.Series({0: 4.8, 1: 4.9}, dtype='float64')
                }
            },
    )
)

# # Add a Energy Storage Component that Requires Materials
# esM.add(
#     fn.Storage(
#         esM=esM,
#         name="Battery",
#         commodity="electricity",
#         chargeEfficiency=0.9,
#         dischargeEfficiency=0.9,
#         materialIntensity={
#             'A': {0: {'copper': 5.2, 'steel': 3.1}, 1: {'copper': 5.3, 'steel': 3.2}},
#             'B': {0: {'copper': 4.8, 'steel': 2.9}, 1: {'copper': 4.9, 'steel': 3.0}}
#             },  # Materials required for commissioning
#         # materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
#     )
# ) 


In [5]:
# Step 3: Add Material Source 
esM.add(
    fn.Source(
        esM=esM, 
        name="Steel Recycling",
        commodity="steel",
        hasCapacityVariable=True,
        material=True,
    )
)

source = esM.add(
    fn.Source(
        esM=esM, 
        name="Copper Recycling",
        commodity="copper",
        hasCapacityVariable=True,
        material=True,
    )
)
#esM.source.commodity

In [6]:
# Step 4: Add Energy Sink Component that consumes Energy 
sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=50,
        
    )
)

In [7]:
# Step 5: Add Material Sink that consumes Materials 
sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Steel demand",
        hasCapacityVariable=False,
        commodity="steel",
        material=True,      
    )
)

In [8]:
esM.aggregateTemporally(numberOfTypicalPeriods=30)


Clustering time series data with 30 typical periods and 24 time steps per period 
further clustered to 12 segments per period...
		(2.7070 sec)



In [9]:
#esM.pyM.op_srcSnk.pprint()

In [10]:
esM.declareOptimizationProblem()

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(1.2484 sec)

Declaring shared potential constraint...
		(0.0001 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.7335 sec)

		(0.0000 sec)

Declaring objective function...
		(8.1908 sec)



In [11]:
# print(list(esM.pyM.op_srcSnk.index_set()))
# esM.pyM.op_srcSnk.pprint()

print(dir(esM.pyM))

['ConstrBigM_srcSnk', 'ConstrCapToNbInt_srcSnk', 'ConstrCapToNbReal_srcSnk', 'ConstrCapacityDevelopment_srcSnk', 'ConstrCapacityMinDec_srcSnk', 'ConstrDesignBinFix_srcSnk', 'ConstrOperation1_srcSnk', 'ConstrOperation2_srcSnk', 'ConstrOperation3_srcSnk', 'ConstrOperation4_srcSnk', 'ConstrOperation5_srcSnk', 'ConstrOperation6_srcSnk', 'ConstrYearlyFullLoadHoursMax_srcSnk', 'ConstrYearlyFullLoadHoursMin_srcSnk', 'ConstrYearlyLimitation_srcSnk', 'ConstraintSharedPotentials', 'DecommConstrCapacityDevelopment_srcSnk', 'DesignLocationComponentVarSet_srcSnk', 'InitialYear_srcSnk', 'Obj', 'Skip', 'StockCommissioning_srcSnk', '_BlockData__autoslot_mappers', '_Block_reserved_words', '_ComponentDataClass', '_DEFAULT_INDEX_CHECKING_ENABLED', '_PPRINT_INDENT', '__auto_slots__', '__autoslot_mappers__', '__class__', '__contains__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getst

In [12]:
esM.optimize()

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 


	declaring constraints... 
		(0.9289 sec)

Declaring shared potential constraint...
		(0.0001 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(1.0659 sec)

		(0.0000 sec)

Declaring objective function...
		(7.9212 sec)

Either solver not selected or specified solver not available.gurobi is set as solver.
Set parameter ServerPassword
Set parameter TSPort to value 41955
Set parameter TokenServer to value "iek3079"
Read LP format model from file /tmp/tmpyxc8h61t.pyomo.lp
Reading time = 0.17 seconds
x1: 105138 rows, 87625 columns, 192756 nonzeros
Set parameter QCPDual to value 1
Set parameter Threads to value 3
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (linux64 - "Rocky Linux 8.8 (Green Obsidian)")

CPU model: Intel(R) Xeon(R) Gold 6154 CPU @ 3.00GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 36 physical cores, 72 logical processors, using up to 3 threads

Optimize a model with 105138 rows, 87625 columns and 192756

In [13]:
esM.getOptimizationSummary("SourceSinkModel", ip=0, outputLevel=0)

A  \
Component        Property                        Unit                     
Copper Recycling NPVcontribution                 [1e9 Euro]         0.0   
                 TAC                             [1e9 Euro/a]       0.0   
                 capacity                        [kg]               0.0   
                 capexCap                        [1e9 Euro/a]       0.0   
                 capexIfBuilt                    [1e9 Euro/a]       NaN   
...                                                                 ...   
Wind Turbines    operation                       [GW*h]        438000.0   
                 opexCap                         [1e9 Euro/a]       0.0   
                 opexIfBuilt                     [1e9 Euro/a]       NaN   
                 opexOp                          [1e9 Euro/a]       0.0   
                 revenueLifetimeShorteningResale [1e9 Euro]           0   

                                                                      B  
Component        Property                        Unit                    
Copper Recycling NPVcontribution                 [1e9 Euro]         0.0  
                 TAC                             [1e9 Euro/a]       0.0  
                 capacity                        [kg]               0.0  
                 capexCap                        [1e9 Euro/a]       0.0  
                 capexIfBuilt                    [1e9 Euro/a]       NaN  
...                                                                 ...  
Wind Turbines    operation                       [GW*h]        438000.0  
                 opexCap                         [1e9 Euro/a]       0.0  
                 opexIfBuilt                     [1e9 Euro/a]       NaN  
                 opexOp                          [1e9 Euro/a]       0.0  
                 revenueLifetimeShorteningResale [1e9 Euro]           0  

[90 rows x 2 columns]

------------------------------------------------------------------------------------------------------------------------
## Test material loop

In [14]:
esM.componentModelingDict.values()

dict_values([<fine.sourceSink.SourceSinkModel object at 0x7fd220dbdd30>])